# Exploratory Geochemical Visualisation

This notebook provides first-pass visual inspection of the compiled dataset before formal data filtering and machine learning. The aim is to visualise broad geochemical patterns using conventional REE and trace element normalised diagrams.

In [ ]:
# =============================================================================
# Environment setup
# =============================================================================

import os
os.environ["OMP_NUM_THREADS"] = "1"

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT_DIR = Path.cwd().parent

DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

DATA_FILE = RAW_DIR / "Data_Input.csv"

EXPORT_DIR = ROOT_DIR / "exports"
FIGURE_DIR = ROOT_DIR / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

print("Environment initialised.")
print(f"Root directory   : {ROOT_DIR}")
print(f"Raw data         : {DATA_FILE}")
print(f"Processed data   : {PROCESSED_DIR}")
print(f"Export directory : {EXPORT_DIR}")
print(f"Figure directory : {FIGURE_DIR}")

## 1. Import Dataset

Load the compiled geochemical dataset and calculate elemental K, P and Ti concentrations from oxide measurements.

In [ ]:
# =============================================================================
# Import dataset
# =============================================================================

df = pd.read_csv(DATA_FILE, encoding="utf8")

df["Ti"] = df["TiO2"] * 0.5994 * 1e4
df["K"] = df["K2O"] * 0.8301 * 1e4
df["P"] = df["P2O5"] * 0.4364 * 1e4

# Validation population
df["Validation_Group"] = np.where(
    df["Country"].eq("China"),
    "China [validation]",
    "Development"
)

# Descriptive plotting groups
df["Plot_Group"] = df["Country"]
df.loc[df["Country"].eq("China"), "Plot_Group"] = "China [validation]"

print("Dataset successfully loaded.")
print(f"Rows   : {len(df)}")
print(f"Columns: {len(df.columns)}")

print("\nValidation population:")
display(
    df[df["Validation_Group"].ne("Development")]
    .groupby(["Series", "Country", "Region"])
    .size()
    .reset_index(name="Analyses")
)

## 2. Define Geochemical Variables

Define rare earth elements, primitive mantle normalised trace element order and metadata columns used for plotting.

In [ ]:
# =============================================================================
# Define geochemical variables
# =============================================================================

REE = [
    "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd",
    "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"
]

TRACE_ELEMENTS = [
    "Cs", "Rb", "Ba", "Th", "U", "Nb", "Ta", "K",
    "La", "Ce", "Pb", "Pr", "Sr", "Nd", "P", "Sm",
    "Zr", "Hf", "Eu", "Ti", "Gd", "Tb", "Dy", "Y",
    "Ho", "Er", "Tm", "Yb", "Lu"
]

META = ["Ref", "Country", "Region", "Locality", "Series"]

print(f"REE elements          : {len(REE)}")
print(f"Trace element sequence: {len(TRACE_ELEMENTS)}")

In [ ]:
# =============================================================================
# Reference values for normalisation
# =============================================================================
# Source: Sun and McDonough (1989)

CI_REE = pd.Series({ "La": 0.237, "Ce": 0.612, "Pr": 0.095, "Nd": 0.467, "Sm": 0.153,
                     "Eu": 0.058, "Gd": 0.2055, "Tb": 0.0374, "Dy": 0.2540, "Ho": 0.0566,
                     "Er": 0.1655, "Tm": 0.0255, "Yb": 0.170, "Lu": 0.0254 })

PM_TRACE = pd.Series({ "Cs": 0.032, "Rb": 0.635, "Ba": 6.989, "Th": 0.085, "U": 0.021,
                       "Nb": 0.713, "Ta": 0.041, "K": 250, "La": 0.687, "Ce": 1.775, 
                       "Pb": 0.185, "Pr": 0.276, "Sr": 21.1, "Nd": 1.354, "P": 95,
                       "Sm": 0.444, "Zr": 11.2, "Hf": 0.309, "Eu": 0.168, "Ti": 1300,
                       "Gd": 0.596, "Tb": 0.108, "Dy": 0.737, "Y": 4.55, "Ho": 0.164,
                       "Er": 0.480, "Tm": 0.074, "Yb": 0.493, "Lu": 0.074 })

print("Reference values loaded from Sun and McDonough (1989).")
print(f"CI REE values: {len(CI_REE)}")
print(f"Primitive mantle values: {len(PM_TRACE)}")

## 3. Dataset Structure

Summarise the distribution of analyses across geological series and countries before plotting. Tibet and Karelia are displayed separately because they are reserved as validation populations.

In [ ]:
# =============================================================================
# Dataset structure
# =============================================================================

country_summary = (
    df.groupby(["Series", "Plot_Group"], dropna=False)
      .size()
      .reset_index(name="Analyses")
      .sort_values(
          ["Series", "Analyses"],
          ascending=[True, False]
      )
)

series_summary = (
    df.groupby("Series")
      .agg(
          Analyses=("Series", "size"),
          Countries=("Country", "nunique")
      )
)

print("=" * 70)
print("SERIES SUMMARY")
print("=" * 70)
display(series_summary)

print("=" * 70)
print("COUNTRY SUMMARY")
print("=" * 70)
display(country_summary)

## 4. Chondrite-normalised REE & Primitive Mantle-normalised Trace Element Diagrams

Rare earth element concentrations are normalised to C1 chondrite and trace element concentrations are normalised to primitive mantle values from Sun and McDonough (1989). Individual analyses are plotted by country, with Tibet and Karelia shown separately as validation groups. For groups with three or more analyses, median values are plotted with shaded envelopes showing the observed minimum to maximum range for each element; groups with fewer than three analyses are represented by the individual analyses only.

In [ ]:
# =============================================================================
# Chondrite-normalised REE patterns by country
# =============================================================================

plot_df = df[["Series", "Plot_Group"] + REE].copy()
plot_df[REE] = plot_df[REE].div(pd.Series(CI_REE))
colours = {"Orogenic": "tab:blue", "Anorogenic": "tab:red"}
x = np.arange(len(REE))

for series, s_df in plot_df.groupby("Series"):
    groups = sorted( [g for g, d in s_df.groupby("Plot_Group") if d[REE].notna().any().any()],
    key=lambda g: ("[validation]" in g, g) )
    ncols, nrows = 4, int(np.ceil(len(groups) / 4))
    fig, axes = plt.subplots(nrows, ncols, figsize=(19, 3.8*nrows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, group in zip(axes, groups):
        v = s_df.loc[s_df["Plot_Group"].eq(group), REE]
        c = colours[series]

        ax.plot(x, v.T, color=c, alpha=0.15, marker="o", ms=2.5, lw=0.7)

        n = v.notna().sum()
        valid = n >= 3
        if valid.any():
            med, lo, hi = v.median(), v.min(), v.max()
            ax.plot(x, med.where(valid), color=c, marker="o", ms=3.5, lw=2)
            ax.fill_between(x, lo.where(valid), hi.where(valid), color=c, alpha=0.15)

        ax.set(title=f"{group}\n$N_{{an}}$={len(v)}", yscale="log",
               xlim=(-0.5, len(REE)-0.5), xticks=x)
        ax.set_xticklabels(REE, rotation=45, ha="right", fontsize=8)
        ax.grid(axis="y", alpha=0.3)

    for ax in axes[len(groups):]:
        ax.remove()

    fig.suptitle(f"{series} lamproites", fontsize=15)
    fig.supylabel("Sample / C1 chondrite", fontsize=12)
    plt.tight_layout()

    outpath = FIGURE_DIR / f"01_REE_chondrite_{series.lower()}_countries.png"
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =============================================================================
# Primitive mantle-normalised trace element patterns by country
# =============================================================================

plot_df = df[["Series", "Plot_Group"] + TRACE_ELEMENTS].copy()
plot_df[TRACE_ELEMENTS] = plot_df[TRACE_ELEMENTS].div(pd.Series(PM_TRACE))
colours = {"Orogenic": "tab:blue", "Anorogenic": "tab:red"}
x = np.arange(len(TRACE_ELEMENTS))

for series, s_df in plot_df.groupby("Series"):
    groups = sorted( [g for g, d in s_df.groupby("Plot_Group") if d[REE].notna().any().any()],
    key=lambda g: ("[validation]" in g, g) )
    ncols, nrows = 3, int(np.ceil(len(groups) / 3))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4*nrows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, group in zip(axes, groups):
        v = s_df.loc[s_df["Plot_Group"].eq(group), TRACE_ELEMENTS]
        c = colours[series]

        ax.plot(x, v.T, color=c, alpha=0.15, marker="o", ms=2.5, lw=0.7)

        n = v.notna().sum()
        valid = n >= 3
        if valid.any():
            med, lo, hi = v.median(), v.min(), v.max()
            ax.plot(x, med.where(valid), color=c, marker="o", ms=3.5, lw=2)
            ax.fill_between(x, lo.where(valid), hi.where(valid),
                            color=c, alpha=0.15)

        ax.set(title=f"{group}\n$N_{{an}}$={len(v)}", yscale="log",
               xlim=(-0.5, len(TRACE_ELEMENTS)-0.5), xticks=x)
        ax.set_xticklabels(TRACE_ELEMENTS, rotation=90, fontsize=7)
        ax.grid(axis="y", alpha=0.3)

    for ax in axes[len(groups):]:
        ax.remove()

    fig.suptitle(f"{series} lamproites", fontsize=15)
    fig.supylabel("Sample / primitive mantle", fontsize=12)
    plt.tight_layout()

    outpath = FIGURE_DIR / f"02_PM_trace_{series.lower()}_countries.png"
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# =============================================================================
# Chondrite-normalised REE patterns by tectonic series and validation group
# =============================================================================

plot_df = df[["Series", "Validation_Group"] + REE].copy()
plot_df[REE] = plot_df[REE].div(pd.Series(CI_REE))
x = np.arange(len(REE))

groups = [
    ("Orogenic", "Development", "Orogenic"),
    ("Orogenic", "China [validation]", "China [validation]"),
    ("Anorogenic", "Development", "Anorogenic"),
    ("Anorogenic", "China [validation]", "China [validation]")
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)

for ax, (series, group, title) in zip(axes.ravel(), groups):
    v = plot_df.loc[
        plot_df["Series"].eq(series) &
        plot_df["Validation_Group"].eq(group), REE
    ]
    c = "tab:blue" if series == "Orogenic" else "tab:red"

    ax.plot(x, v.T, color=c, alpha=0.12, marker="o", ms=2.5, lw=0.7)

    valid = v.notna().sum() >= 3
    med, lo, hi = v.median(), v.min(), v.max()
    ax.plot(x, med.where(valid), color=c, marker="o", ms=3.5, lw=2)
    ax.fill_between(x, lo.where(valid), hi.where(valid), color=c, alpha=0.15)

    ax.set(title=f"{title}\n$N_{{an}}$={len(v)}", yscale="log",
           xlim=(-0.5, len(REE)-0.5), xticks=x)
    ax.set_xticklabels(REE, rotation=45, ha="right", fontsize=8)
    ax.grid(axis="y", alpha=0.3)

fig.supylabel("Sample / C1 chondrite", fontsize=12)
plt.tight_layout()

outpath = FIGURE_DIR / "03_REE_chondrite_series.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# Primitive mantle-normalised trace element patterns by tectonic series
# =============================================================================

plot_df = df[["Series", "Validation_Group"] + TRACE_ELEMENTS].copy()
plot_df[TRACE_ELEMENTS] = plot_df[TRACE_ELEMENTS].div(pd.Series(PM_TRACE))
x = np.arange(len(TRACE_ELEMENTS))


groups = [
    ("Orogenic", "Development", "Orogenic"),
    ("Orogenic", "China [validation]", "China [validation]"),
    ("Anorogenic", "Development", "Anorogenic"),
    ("Anorogenic", "China [validation]", "China [validation]")
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)

for ax, (series, group, title) in zip(axes.ravel(), groups):
    v = plot_df.loc[
        plot_df["Series"].eq(series) &
        plot_df["Validation_Group"].eq(group), TRACE_ELEMENTS
    ]
    c = "tab:blue" if series == "Orogenic" else "tab:red"

    ax.plot(x, v.T, color=c, alpha=0.12, marker="o", ms=2, lw=0.6)

    valid = v.notna().sum() >= 3
    med, lo, hi = v.median(), v.min(), v.max()
    ax.plot(x, med.where(valid), color=c, marker="o", ms=3, lw=2)
    ax.fill_between(x, lo.where(valid), hi.where(valid), color=c, alpha=0.15)

    ax.set(title=f"{title}\n$N_{{an}}$={len(v)}", yscale="log",
           xlim=(-0.5, len(TRACE_ELEMENTS)-0.5), xticks=x)
    ax.set_xticklabels(TRACE_ELEMENTS, rotation=90, fontsize=7)
    ax.grid(axis="y", alpha=0.3)

fig.supylabel("Sample / primitive mantle", fontsize=12)
plt.tight_layout()

outpath = FIGURE_DIR / "04_PM_trace_series.png"
plt.savefig(outpath, dpi=300, bbox_inches="tight")
plt.show()